# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U openai==2.36.0 sounddevice==0.5.5 dotenv==0.9.9 azure-ai-voicelive==1.2.0

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_URI = os.getenv("AZURE_OPENAI_URI")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_REALTIME_DEPLOYMENT = os.getenv("AZURE_OPENAI_REALTIME_DEPLOYMENT")

# Voice models
Microsoft foundry 는 다양한 voice models 을 제공한다. Microsoft 에서 직접 개발하는 MAI 모델이나 OpenAI 의 gpt 모델 등이 있는데, 사용자 어플리케이션은 최적화되어 있는 모델들이 필요하고 그 목적도 다양하다. Text-to-speech, speech-to-text 그리고 realtime 모델 등에 걸쳐 다양한 voice models 을 Microsoft foundry 에서 만나보자.

## gpt-realtime
OpenAI 가 개발한 실시간 음성-음성 대화형 AI 모델이다. STT 나 TTS 의 중간 과정없이 voice 인터페이스를 제공하고 사람처럼 자연스러운 대화가 가능하다.
아래 코드는 low latency 와 interruption 기능이 구현되어 있는 간단한 voice 챗봇이다. 실행시켜 마이크로 대화해보자.

### 동작 원리 (블록도)

```text
   🎙️ 마이크 (PCM 24kHz)
        │
        │  ▶ input_audio_buffer.append      (client ──WS──▶ server)
        ▼
  ┌─ gpt-realtime (서버) ───────────────────────────────────────
  │   ① Server VAD     : 발화 시작/종료를 감지해 대화 턴을 분리
  │   ② Speech-to-Speech: 음성을 직접 이해·추론하고 음성으로 직접
  │                       생성 (STT·TTS 중간 변환 단계가 없음)
  │   ③ Transcript     : 입력/출력 텍스트도 함께 스트리밍
  └──────────────────────────────────────────────────────────────
        │
        │  ◀ response.output_audio.delta    (server ──WS──▶ client)
        ▼
   🔊 스피커 (PCM 24kHz)

   ↕ 전 구간이 하나의 WebSocket 양방향(full-duplex) 스트림
     → 낮은 지연(low latency) + 말하는 중 끼어들기(interruption) 가능
```

**아래 코드의 이벤트가 위 흐름에 그대로 대응한다.**

| 단계 | Realtime 이벤트 |
|------|-----------------|
| 마이크 → 서버 전송 | `input_audio_buffer.append` |
| 사용자 발화 감지 | `input_audio_buffer.speech_started` |
| 응답 오디오 수신(재생) | `response.output_audio.delta` |
| 음성 인식 결과 | `conversation.item.input_audio_transcription.completed` |
| 응답 종료 | `response.done` |


In [ ]:
import os
import asyncio
import base64
import threading

import sounddevice as sd
from openai import AsyncOpenAI

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 간단한 스피커 래퍼 클래스
class Speaker:
    def __init__(self):
        self.buf = bytearray()
        self.lock = threading.Lock()
        self.stream = sd.RawOutputStream(
            samplerate=24000, channels=1, dtype="int16", blocksize=0, callback=self._cb,
        )

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        # PortAudio 콜백은 별도의 스레드에서 호출되므로, 버퍼 접근을 락으로 보호
        with self.lock:
            avail = min(need, len(self.buf))
            outdata[:avail] = bytes(self.buf[:avail])
            del self.buf[:avail]
            if avail < need:
                outdata[avail:] = b"\x00" * (need - avail)

    def add(self, pcm: bytes):
        # PCM 데이터를 버퍼에 추가하는 메서드. 콜백과 동시에 호출될 수 있으므로 락으로 보호
        with self.lock:
            self.buf.extend(pcm)

    def clear(self):
        # 버퍼를 초기화하는 메서드. 콜백과 동시에 호출될 수 있으므로 락으로 보호
        with self.lock:
            self.buf.clear()

    def start(self):
        self.stream.start()

    def close(self):
        try:
            self.stream.stop()
        finally:
            self.stream.close()


# PortAudio 초기화 및 재설정
reset_portaudio()

# OpenAI 클라이언트 초기화
client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.replace("https://", "wss://").rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)
speaker, mic, sender = None, None, None
try:
    # 스피커와 마이크 초기화
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    # 마이크 콜백 함수: PCM 데이터를 받아서 asyncio 큐에 넣음
    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    # 마이크 스트림 설정: 24000Hz, 모노, 16비트 PCM, 콜백 함수 등록
    mic = sd.RawInputStream(
        samplerate=24000, channels=1, dtype="int16", blocksize=0, callback=mic_cb,
    )

    # OpenAI 실시간 연결 설정 및 이벤트 루프
    async with client.realtime.connect(model=AZURE_OPENAI_REALTIME_DEPLOYMENT) as conn:
        await conn.session.update(session={
            "type": "realtime",
            "instructions": "너는 친절한 한국어 AI 비서야. 사용자의 음성 메시지를 받아서 대답해줘.",
            "output_modalities": ["audio"],
            "audio": {
                "input": {
                    "format": {"type": "audio/pcm", "rate": 24000},
                    "transcription": {"model": "gpt-4o-transcribe", "language": "ko"}, # 실시간 음성 인식
                    "turn_detection": {
                        "type": "server_vad",   # 서버에서 음성 활동 감지(VAD) 사용
                        "threshold": 0.5,   # VAD 민감도 조절 (0.0 ~ 1.0, 낮을수록 민감)
                        "prefix_padding_ms": 300,   # 음성 시작 전 패딩
                        "silence_duration_ms": 500, # 음성 종료로 간주할 무음 지속 시간
                        "create_response": True,
                        "interrupt_response": True, # 음성 인식 중간에 응답 생성 허용
                    },
                },
                "output": {
                    "voice": "alloy",
                    "format": {"type": "audio/pcm", "rate": 24000},
                },
            },
        })
        speaker.start()
        mic.start()
        print("🎙️  말해보세요. (Ctrl+C 로 종료)\n")

        # 마이크에서 PCM 데이터를 읽어서 OpenAI로 전송하는 비동기 태스크
        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                await conn.input_audio_buffer.append(
                    audio=base64.b64encode(chunk).decode(),
                )

        sender = asyncio.create_task(pump_mic())
        # OpenAI 이벤트 처리 루프
        async for event in conn:
            # 이벤트 타입에 따라 스피커에 PCM 데이터를 추가하거나, 인식된 텍스트를 출력하거나, 에러를 처리
            if event.type == "response.output_audio.delta":
                speaker.add(base64.b64decode(event.delta))
            # 서버 VAD가 음성 활동을 감지하여 turn이 시작되었을 때, 이전 대화 내용을 초기화
            elif event.type == "response.output_audio_transcript.delta":
                print(event.delta, end="", flush=True)
            # 서버 VAD가 음성 활동이 종료되었다고 판단하여 turn이 완료되었을 때, 최종 대화 내용을 출력하고 스피커 버퍼 초기화
            elif event.type == "conversation.item.input_audio_transcription.completed":
                print(f"\n🧑 {event.transcript}")
            # 서버 VAD가 음성 활동이 시작되었다고 판단하여 turn이 시작되었을 때, 스피커 버퍼 초기화
            elif event.type == "input_audio_buffer.speech_started":
                print("\n🎧 음성 인식 시작...")
                speaker.clear()
            # 서버 VAD가 음성 활동이 종료되었다고 판단하여 turn이 완료되었을 때, 최종 대화 내용을 출력하고 스피커 버퍼 초기화
            elif event.type == "response.done":
                print("\n✅ 응답 완료")
                print()
            elif event.type == "error":
                print("\n⚠️  ERROR:", event.error)
finally:
    if sender:
        sender.cancel()
    if mic:
        try:
            mic.stop(); mic.close()
        except Exception as e:
            print(f"마이크 종료 에러: {e}")
    if speaker:
        try:
            speaker.close()
        except Exception as e:
            print(f"스피커 종료 에러: {e}")

## Voice live

Azure Microsoft Foundry Voice Live API 는 gpt-realtime 같은 음성-음성(speech-to-speech) 모델을 하나의 WebSocket 엔드포인트로 감싸고, 여기에 서버측 음질 향상 (에코 제거·딥 노이즈 억제), Azure 뉴럴 보이스, (선택) 아바타·표정(viseme) 까지 얹어주는 통합 음성 대화 서비스다. 앞의 gpt-realtime 셀을 직접 WebSocket 으로 다룬 것과 달리, `azure-ai-voicelive` SDK 의 `connect()` 한 번으로 동일한 음성 챗봇을 더 간단하게 만들 수 있다.

### 동작 원리 (블록도)

```text
   🎙️ 마이크 (PCM16 24kHz)
        │  input_audio_buffer.append (base64)
        ▼
  ╔═ Azure Microsoft Foundry · Voice Live API (WebSocket) ════════
  ║   ① 입력 전처리   : 에코 제거 + 딥 노이즈 억제 (서버측)
  ║   ② Server VAD    : 발화 시작/종료 감지 → 턴 분리
  ║   ③ gpt-realtime  : 음성을 직접 이해·추론·생성 (speech-to-speech)
  ║   ④ 출력          : Azure 뉴럴 보이스 TTS + 전사(transcript)
  ╚═══════════════════════════════════════════════════════════════
        │  response.audio.delta (PCM bytes)
        ▼
   🔊 스피커 (PCM16 24kHz)
```

### gpt-realtime(직접 WebSocket) 과 비교

| 구분 | gpt-realtime (앞 셀) | Voice Live API |
|------|----------------------|----------------|
| 사용법 | OpenAI realtime WS 직접 제어 | `azure-ai-voicelive` SDK `connect()` |
| 음질 처리 | 직접 구현(지터버퍼 등) | 에코 제거·노이즈 억제 서버 기본 제공 |
| 보이스 | OpenAI 보이스(alloy 등) | OpenAI 보이스 + Azure 뉴럴 보이스 |
| 확장 | 음성만 | 아바타·표정(viseme) 등 옵션 |

In [ ]:
import asyncio
import base64
import threading

import sounddevice as sd
from azure.core.credentials import AzureKeyCredential
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import (
    RequestSession,
    Modality,
    InputAudioFormat,
    OutputAudioFormat,
    ServerVad,
    ServerEventType,
    AzureStandardVoice,
    AudioEchoCancellation,
    AudioNoiseReduction,
)

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 지터 버퍼 스피커 (재생 끊김 완화)
class Speaker:
    def __init__(self, prebuffer_ms=150):
        self.buf = bytearray()
        self.lock = threading.Lock()
        self.playing = False
        self.prebuffer = int(24000 * 2 * prebuffer_ms / 1000)  # 24kHz·16bit 기준 바이트
        self.stream = sd.RawOutputStream(
            samplerate=24000, channels=1, dtype="int16",
            blocksize=0, latency="high", callback=self._cb,
        )

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        with self.lock:
            if not self.playing:  # 선버퍼링: 충분히 쌓이기 전엔 무음
                if len(self.buf) < self.prebuffer:
                    outdata[:] = b"\x00" * need
                    return
                self.playing = True
            avail = min(need, len(self.buf))
            outdata[:avail] = bytes(self.buf[:avail])
            del self.buf[:avail]
            if avail < need:
                outdata[avail:] = b"\x00" * (need - avail)
                self.playing = False  # 언더런 → 다시 선버퍼링

    def add(self, pcm: bytes):
        with self.lock:
            self.buf.extend(pcm)

    def clear(self):
        with self.lock:
            self.buf.clear()
            self.playing = False

    def start(self):
        self.stream.start()

    def close(self):
        try:
            self.stream.stop()
        finally:
            self.stream.close()

reset_portaudio()
speaker, mic, sender = None, None, None
try:
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    mic = sd.RawInputStream(
        samplerate=24000, channels=1, dtype="int16",
        blocksize=2400, latency="high", callback=mic_cb,
    )

    # Voice Live 연결 + 세션 설정
    async with connect(
        endpoint=AZURE_OPENAI_URI,
        credential=AzureKeyCredential(AZURE_OPENAI_API_KEY),
        model=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ) as conn:
        # 세션 설정 — gpt-realtime 셀과 달리 에코 제거/노이즈 억제를 서버가 처리
        await conn.session.update(session=RequestSession(
            modalities=[Modality.TEXT, Modality.AUDIO],
            instructions="너는 친절한 한국어 AI 비서야. 음성으로 자연스럽고 간결하게 답해줘.",
            voice=AzureStandardVoice(name="ko-kr-SunHi:DragonHDLatestNeural"),
            input_audio_format=InputAudioFormat.PCM16,
            output_audio_format=OutputAudioFormat.PCM16,
            turn_detection=ServerVad(
                threshold=0.5, prefix_padding_ms=300, silence_duration_ms=500,
            ),
            input_audio_echo_cancellation=AudioEchoCancellation(),
            input_audio_noise_reduction=AudioNoiseReduction(type="azure_deep_noise_suppression"),
        ))
        speaker.start()
        mic.start()
        print("🎙️  말해보세요. (Ctrl+C 로 종료)\n")

        # 마이크 PCM → base64 로 인코딩해 서버로 스트리밍
        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                await conn.input_audio_buffer.append(audio=base64.b64encode(chunk).decode())

        sender = asyncio.create_task(pump_mic())
        # Voice Live 이벤트 처리 루프
        async for event in conn:
            # 주의: Voice Live 의 오디오 delta 는 이미 디코딩된 bytes (raw realtime 의 base64 와 다름)
            if event.type == ServerEventType.RESPONSE_AUDIO_DELTA:
                speaker.add(event.delta)
            # AI 응답 전사(텍스트) 스트리밍
            elif event.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DELTA:
                print(event.delta, end="", flush=True)
            # 사용자 음성 인식 결과
            elif event.type == ServerEventType.CONVERSATION_ITEM_INPUT_AUDIO_TRANSCRIPTION_COMPLETED:
                print(f"\n🧑 {event.transcript}")
            # 사용자가 말하기 시작 → 재생 중단(끼어들기)
            elif event.type == ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED:
                print("\n🎧 음성 인식 시작...")
                speaker.clear()
            # 응답 완료
            elif event.type == ServerEventType.RESPONSE_DONE:
                print("\n✅ 응답 완료\n")
            elif event.type == ServerEventType.ERROR:
                print("\n⚠️  ERROR:", event.error)
finally:
    if sender:
        sender.cancel()
    if mic:
        try:
            mic.stop(); mic.close()
        except Exception as e:
            print(f"마이크 종료 에러: {e}")
    if speaker:
        try:
            speaker.close()
        except Exception as e:
            print(f"스피커 종료 에러: {e}")